In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
print("TensorFlow version:", tf.__version__)
# Load the TensorBoard notebook extension
%load_ext tensorboard
import datetime
import shutil

TensorFlow version: 2.18.0


In [2]:
%reload_ext tensorboard
from tensorflow.keras import Model
from pathlib import Path
import pandas as pd

In [3]:
BATCH_SIZE = 64   
BUFFER_SIZE = 1000   
LEARNING_RATE = 0.00005  
EPOCHS = 600 #400
# best results 
# - 
# - Epoch 600, Loss: 18179076.0, Accuracy: 2283.99658203125, Test Loss: 25707932.0, Test MAE: 2427.27978515625


In [4]:
input_dir = Path('./data/prepared')
logs_path = Path('./data/logs')
if logs_path.exists():
  shutil.rmtree(logs_path) # удаляем, если существует /logs
logs_path.mkdir(parents=True)

X_train_name = input_dir / 'X_train.csv'
y_train_name = input_dir / 'y_train.csv'
X_test_name = input_dir / 'X_test.csv'
y_test_name = input_dir / 'y_test.csv'

X_train = pd.read_csv(X_train_name)
y_train = pd.read_csv(y_train_name)
X_test = pd.read_csv(X_test_name)
y_test = pd.read_csv(y_test_name)

X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()

# print(X_train.isnull().sum())
# print(y_train.isnull().sum())
# print(X_train.dtypes)  # Типы данных в X_train
# print(y_train.dtypes)  # Типы данных в y_train


y_train = pd.read_csv(y_train_name)
y_test = pd.read_csv(y_test_name)

train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE)

In [5]:
@tf.keras.utils.register_keras_serializable() #  Декоратор позволяет сериализовать и десериализовать модель для сохранения и загрузки.
class SomeModel(Model):
    def __init__(self, neurons_cnt=64, **kwargs):
        super(SomeModel, self).__init__(**kwargs)
        self.neurons_cnt = neurons_cnt  # Сохраняем значение параметра для конфигурации
        self.d_in = Dense(27, activation='relu')
        self.d1 = Dense(neurons_cnt, activation='relu')
        self.d2 = Dense(neurons_cnt, activation='relu')
        self.d3 = Dense(neurons_cnt, activation='relu')
        self.d_out = Dense(1)

    def call(self, x):
        x = self.d_in(x)
        x = self.d1(x)
        x = self.d2(x)
        x = self.d3(x)
        return self.d_out(x)
         
    def build(self, input_shape): # надо явно определить для построения
        super(SomeModel, self).build(input_shape)
        
    def get_config(self): 
        # Возвращаем параметры модели, включая кастомные
        config = super(SomeModel, self).get_config()
        config.update({
            "neurons_cnt": self.neurons_cnt  # Добавляем кастомный параметр в конфигурацию
        })
        return config

    @classmethod
    def from_config(cls, config):
        # Создаём экземпляр класса из конфигурации
        return cls(**config)

In [6]:
# Create an instance of the model
model = SomeModel(neurons_cnt=64) # 32
model.build(input_shape=(None, 8))  # 16

In [7]:
loss_object = tf.keras.losses.MeanSquaredError() # что? 
optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)

train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.MeanAbsoluteError(name='train_mae')

test_loss = tf.keras.metrics.Mean(name='test_loss')
test_accuracy = tf.keras.metrics.MeanAbsoluteError(name='test_mae')

In [8]:
@tf.function
def train_step(input_vector, labels):
  with tf.GradientTape() as tape:
    # training=True is only needed if there are layers with different
    # behavior during training versus inference (e.g. Dropout).
    predictions = model(input_vector, training=True)
    loss = loss_object(labels, predictions)
  gradients = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))

  train_loss(loss)
  train_accuracy(labels, predictions)

@tf.function
def test_step(input_vector, labels):
  # training=False is only needed if there are layers with different
  # behavior during training versus inference (e.g. Dropout).
  predictions = model(input_vector, training=False)
  t_loss = loss_object(labels, predictions)

  test_loss(t_loss)
  test_accuracy(labels, predictions)

In [9]:
from tensorflow import keras
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
train_log_dir = logs_path / 'gradient_tape' / current_time / 'train'
train_log_dir.mkdir(exist_ok=True, parents=True)
test_log_dir = logs_path / 'gradient_tape' / current_time / 'test'
test_log_dir.mkdir(exist_ok=True, parents=True)
train_summary_writer = tf.summary.create_file_writer(str(train_log_dir))
test_summary_writer = tf.summary.create_file_writer(str(test_log_dir))

logdir=logs_path / "fit" / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
logdir.mkdir(exist_ok=True, parents=True)
fit_summary_writer = tf.summary.create_file_writer(str(logdir))

tf.summary.trace_on(graph=True, profiler=True, profiler_outdir=str(logdir))

for epoch in range(EPOCHS):
  # Reset the metrics at the start of the next epoch
  for (x_train, y_train) in train_ds:

    with fit_summary_writer.as_default():
      train_step(x_train, y_train)


  with train_summary_writer.as_default():
    tf.summary.scalar('loss', train_loss.result(), step=epoch)
    tf.summary.scalar('accuracy', train_accuracy.result(), step=epoch)

  for (x_test, y_test) in test_ds:
    test_step(x_test, y_test)

  with test_summary_writer.as_default():
    tf.summary.scalar('loss', test_loss.result(), step=epoch)
    tf.summary.scalar('mae', test_accuracy.result(), step=epoch)

  template = 'Epoch {}, Loss: {}, Accuracy: {}, Test Loss: {}, Test MAE: {}'
  print (template.format(epoch+1,
                         train_loss.result(),
                         train_accuracy.result(),
                         test_loss.result(),
                         test_accuracy.result()))

  # Reset metrics every epoch
  train_loss.reset_state()
  test_loss.reset_state()
  train_accuracy.reset_state()
  test_accuracy.reset_state()

with fit_summary_writer.as_default():
  tf.summary.trace_export(
      name="my_func_trace",
      step=0,
      profiler_outdir=str(logdir)
  )

Epoch 1, Loss: 0.08687131851911545, Accuracy: 0.21534661948680878, Test Loss: 0.07806190103292465, Test MAE: 0.20498211681842804
Epoch 2, Loss: 0.08515502512454987, Accuracy: 0.21176359057426453, Test Loss: 0.07646475732326508, Test MAE: 0.2015332281589508
Epoch 3, Loss: 0.0833968073129654, Accuracy: 0.20822936296463013, Test Loss: 0.0748918205499649, Test MAE: 0.19815672934055328
Epoch 4, Loss: 0.08172444254159927, Accuracy: 0.2047676146030426, Test Loss: 0.07338669896125793, Test MAE: 0.1948881596326828
Epoch 5, Loss: 0.08009332418441772, Accuracy: 0.2013828009366989, Test Loss: 0.07190819084644318, Test MAE: 0.19166545569896698
Epoch 6, Loss: 0.07855745404958725, Accuracy: 0.19807258248329163, Test Loss: 0.07047843933105469, Test MAE: 0.18853753805160522
Epoch 7, Loss: 0.07702711969614029, Accuracy: 0.1949176788330078, Test Loss: 0.06905746459960938, Test MAE: 0.18541546165943146
Epoch 8, Loss: 0.0754973441362381, Accuracy: 0.19190135598182678, Test Loss: 0.06770128756761551, Test M

In [10]:
model.save('./data/models/mymodel.keras')
loaded_model = keras.models.load_model('./data/models/mymodel.keras')
loaded_model

<SomeModel name=some_model, built=True>

In [11]:
model

<SomeModel name=some_model, built=True>

In [12]:
%tensorboard --logdir ./data/logs

Reusing TensorBoard on port 6007 (pid 12116), started 12:50:09 ago. (Use '!kill 12116' to kill it.)

In [13]:
!kill 12116

"kill" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.
